# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [42]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-02-19T22:06:07.067201",
    "last_interaction": "2026-02-19T22:06:32.587880",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-02-19T22:06:09.534336",
    "last_interaction": "2026-02-19T22:06:40.200595",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:a05b22a4-4e3f-4069-9d54-3dd64fd0ba1a",
    "dctIssued": "2026-02-19T22:06:07.100387Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [43]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:f24549d4-3a08-456e-8fb4-b16e489ecd84",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:46e0

## Provider creates initial offer (Provider -> Consumer)

In [44]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:f24549d4-3a08-456e-8fb4-b16e489ecd84",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e"
    },
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1579be6c-3883-412d-a7

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [45]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:f24549d4-3a08-456e-8fb4-b16e489ecd84",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e"
    },
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:46e0b5aa-3d30-4622-8678-77116c1980c8",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [46]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:f24549d4-3a08-456e-8fb4-b16e489ecd84",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e"
    },
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1579be6c-3883-412d-a709-b57c3b577050",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [47]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:46e0b5aa-3d30-4622-8678-77116c1980c8",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T11:04:43.899027Z",
    "updatedAt": "2026-02-22T11:04:45.140641Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:b

## Provider creates the Agreement (Provider -> Consumer)

In [48]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1579be6c-3883-412d-a709-b57c3b577050",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T11:04:43.817159Z",
    "updatedAt": "2026-02-22T11:04:45.479080Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:b9e18

## Consumer verifies the agreement (Consumer -> Provider)

In [49]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:46e0b5aa-3d30-4622-8678-77116c1980c8",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T11:04:43.899027Z",
    "updatedAt": "2026-02-22T11:04:45.827254Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:b

## Provider finalizes the negotiation (Provider -> Consumer)

In [50]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b9e1862f-67e1-4264-8802-4207b9b797eb",
    "providerPid": "urn:provider-pid:2aa6ffe1-afa0-4964-9ec2-336308b78d18",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1579be6c-3883-412d-a709-b57c3b577050",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T11:04:43.817159Z",
    "updatedAt": "2026-02-22T11:04:46.196515Z",
    "identifiers": {
      "providerPid": "urn:provider-pid

## Final agreement

In [51]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:fdc5883b-5df7-4479-8536-4df6a2446200",
  "negotiationAgentProcessId": "urn:negotiation-process:1579be6c-3883-412d-a709-b57c3b577050",
  "negotiationAgentMessageId": "urn:negotiation-message:e23b8e77-f2a0-4a7f-bc9e-044feeab14cd",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:fdc5883b-5df7-4479-8536-4df6a2446200",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e",
    "timestamp": "1771758285"
  },
  "target": "urn:dataset:2ba00792-8070-48de-ad65-1d8bb1f6483e",
  "state": "ACTIVE",
  "createdAt": "2026-02-22T11:04:45.484823Z",
  "updatedAt": "2026-02-22T11:04:46.203029Z"
}

Final agreement id: 
urn:agreement:fdc5883b-5df7-4479-8536-4df6a2446200



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [52]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:fdc5883b-5df7-4479-8536-4df6a2446200",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:a0216149-80fd-4cdb-95df-9abaa5a22906",
    "providerPid": "urn:provider-pid:e509e181-88c7-4150-931d-9bf7f3bd4108",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:84a345a2-e31c-427c-9094-d4a62d85b9ae",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fdc5883b-5df7-4479-8536-4df6a2446200",
    "callbackAddress": "http://127.0.0.1:

## Start transfer (Provider -> Consumer)

In [53]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:a0216149-80fd-4cdb-95df-9abaa5a22906",
    "providerPid": "urn:provider-pid:e509e181-88c7-4150-931d-9bf7f3bd4108"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:a0216149-80fd-4cdb-95df-9abaa5a22906",
    "providerPid": "urn:provider-pid:e509e181-88c7-4150-931d-9bf7f3bd4108",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:0ef486b7-bf9b-4f8a-8e10-549b9c12caa7",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:10151320-3bbe-4c5a-a94c-0bc5546c37a7",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:fdc5883b

## Suspend transfer (Consumer -> Provider)

In [13]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:defdfd5a-c70e-4205-bb60-b71e30c4ce75",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:9a8a95e4-9115-4d4a-a559-71a4e5bd9927",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
   

## Restart transfer (Consumer -> Provider)

In [14]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1200/dataplane/proxy/urn:dataplane-transfer:7da5b0ae-8c4a-48b1-a265-9f76fc09df78",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:defdfd5a-c70e-4205-bb60-b71e30c4ce75",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:9a8a95e

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [15]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:20f33b28-9da2-4d5b-b18b-77f0aa6af8c1",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:9a8a95e4-9115-4d4a-a559-71a4e5bd9927",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
   

## Failure Test: Attempt start with invalid parameters

In [16]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698"
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "HTTP Error 400 Bad Request: {\"@context\":[\"https://w3id.org/dspace/2025/1/context.jsonld\"],\"@type\":\"TransferError\",\"consumerPid\":null,\"providerPid\":null,\"code\":\"6030\",\"reason\":[\"TransferProcessMessageType TransferStartMessage is not allowed here. Current state is SUSPENDED ByProvider\",\"Failed to parse file\"]}"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [17]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "6030",
    "reason": [
      "TransferProcessMessageType TransferSuspensionMessage is not allowed here. Current state is SUSPENDED",
      "Failed to parse file"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [18]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1ac97291-d297-4569-9a6c-ea5867ec7a0d",
    "providerPid": "urn:provider-pid:c3155a4a-4383-44fd-bda3-e595f9e3e698",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:20f33b28-9da2-4d5b-b18b-77f0aa6af8c1",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:9a8a95e4-9115-4d4a-a559-71a4e5bd9927",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T10:06:04.518284Z",